In [1]:
%pip install pandas numpy openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import numpy as np
import pandas as pd
from datetime import datetime

RAW_FILE = 'Raw_Dataset.xlsx'
CLEAN_FILE = 'Cleaned_Dataset_Output.xlsx'
REFERENCE_FILE = 'Cleaned_Dataset.xlsx'

if not os.path.exists(RAW_FILE):
    raise FileNotFoundError(
        f'{RAW_FILE} not found in folder {os.getcwd()}')

df = pd.read_excel(RAW_FILE,
                   sheet_name=0,
                   dtype=str,
                   keep_default_na=False)
print('Rows loaded:', df.shape[0],
      '| Columns:', df.shape[1])



Rows loaded: 4000 | Columns: 16


In [3]:
# STEP 1: STANDARDIZE COLUMN NAMES
COL_MAP = {'record_id': 'Record_ID',
           'report_date': 'Report_Date',
           'fiscal_quarter': 'Fiscal_Quarter',
           'department': 'Department',
           'cost_center': 'Cost_Center',
           'region': 'Region',
           'category': 'Category',
           'account_head': 'Account_Head',
           'expense_type': 'Expense_Type',
           'actual_amount_inr': 'Actual_Amount',
           'budget_amount_inr': 'Budget_Amount',
           'customer_count': 'Customer_Count',
           'payment_status': 'Payment_Status',
           'source_system': 'Source_System',
           'approved_by': 'Approved_By',
           'remarks': 'Remarks'}

def simplify(name):
    name = name.strip().lower()
    name = re.sub(r'[^a-z0-9]+', '_', name)
    return name.strip('_')

df.columns = [COL_MAP[simplify(c)]
              for c in df.columns]
print(df.columns.tolist())

['Record_ID', 'Report_Date', 'Fiscal_Quarter', 'Department', 'Cost_Center', 'Region', 'Category', 'Account_Head', 'Expense_Type', 'Actual_Amount', 'Budget_Amount', 'Customer_Count', 'Payment_Status', 'Source_System', 'Approved_By', 'Remarks']


In [4]:
#STEP 2: REMOVE JUNK ROWS
blank_rows = (df.apply(lambda s: s.str.strip() == '')
              .all(axis=1))
header_rows = (df['Record_ID'].str.strip().str.lower()
               .isin(['record id', 'record_id']))
total_rows = (df['Record_ID'].str.contains('total', case=False)
              | df['Department'].str.contains('total', case=False))
df = df[~(blank_rows | header_rows | total_rows)].copy()
print('After removing junk rows:', df.shape[0])


After removing junk rows: 3965


In [5]:
# STEP 3: REPLACE PLACEHOLDER TOKENS WITH NAN

PLACEHOLDERS = {'', 'n/a', 'na', 'null', 'none', '-', '--',
                '?', 'missing', 'nan', 'nil'}
df = df.apply(lambda s: s.where(
    ~s.str.strip().str.lower().isin(PLACEHOLDERS), np.nan))
print('Missing values per column:')
print(df.isnull().sum())

Missing values per column:
Record_ID           16
Report_Date         16
Fiscal_Quarter     113
Department           0
Cost_Center        111
Region             114
Category            88
Account_Head         0
Expense_Type      1358
Actual_Amount       63
Budget_Amount      157
Customer_Count    2112
Payment_Status     108
Source_System        0
Approved_By        224
Remarks           3394
dtype: int64


In [6]:
# STEP 4: TRIM AND COLLAPSE WHITESPACE

df = df.apply(lambda s: s.str.strip()
              .str.replace(r'\s+', ' ', regex=True))

In [7]:
# STEP 5: NORMALIZE LETTER CASE
# Reused at Step-23
CAT_COLS = ['Department', 'Region', 'Category', 'Account_Head',
            'Expense_Type', 'Payment_Status', 'Source_System']
for col in CAT_COLS:
    df[col] = df[col].str.lower()
df['Record_ID'] = df['Record_ID'].str.upper()


In [8]:
# STEP 6: MAP VARIANTS TO CANONICAL LABELS

def build_map(spec):
    result = {}
    for canonical, variants in spec.items():
        result[canonical.lower()] = canonical
        for v in variants:
            result[v.lower()] = canonical
    return result

DEPT_MAP = build_map({
    'Sales': ['sales dept', 'sales department', 'slaes'],
    'Marketing': ['mktg', 'marketng'],
    'HR': ['h.r.', 'human resources', 'hr dept'],
    'Operations': ['ops', 'operatons'],
    'IT': ['i.t.', 'information technology', 'it dept'],
    'Finance': ['fin', 'finanace'],
    'Customer Support': ['cust support', 'customer service',
                         'cust. support'],
    'R&D': ['r and d', 'research & development', 'rnd']})
REGION_MAP = build_map({'North': ['n', 'nth'],
                            'South': ['s', 'sth'],
                            'East': ['e'], 'West': ['w'],
                            'Central': ['cntrl', 'ctr']})
CATEGORY_MAP = build_map({
    'Revenue': ['rev', 'income', 'revenues'],
    'Expense': ['exp', 'expenses', 'cost']})
EXPENSE_TYPE_MAP = build_map({
    'COGS': ['direct', 'direct cost', 'cost of goods sold'],
    'Operating': ['opex', 'operating expense', 'indirect']})
PAYMENT_MAP = build_map({'Paid': ['complete', 'settled'],
                             'Pending': ['pndg', 'in process'],
                             'Overdue': ['over due', 'od']})
SOURCE_MAP = build_map({'Excel': ['ms excel', 'xlsx'],
                            'CSV': ['csv file'],
                            'SQL Database': ['sql db', 'sql', 'mysql'],
                            'ERP': ['erp system', 'sap erp']})
ACCOUNT_MAP = build_map({
    'Product Sales': ['prod sales'],
    'Subscription Revenue': ['subscription rev', 'subscriptions'],
    'Service Revenue': ['service income'],
    'Licensing Income': ['licence income', 'license income'],
    'Raw Materials': ['raw material'],
    'Direct Labour': ['direct labor'],
    'Freight & Logistics': ['freight and logistics', 'logistics'],
    'Salaries': ['salary', 'salaries & wages'],
    'Rent': ['office rent'],
    'Marketing Spend': ['mktg spend'],
    'Software Licences': ['software licenses', 'sw licences'],
    'Travel': ['travel & entertainment', 't&e'],
    'Utilities': ['utility bills'],
    'Maintenance': ['repairs & maintenance'],
    'Training': ['training & development'],
    'Recruitment': ['hiring costs'],
    'R&D Expenses': ['r and d expenses', 'rnd expenses'],
    'Legal & Compliance': ['legal and compliance'],
    'Support Tools': ['support software']})

MAPS = {'Department': DEPT_MAP, 'Region': REGION_MAP,
        'Category': CATEGORY_MAP, 'Account_Head': ACCOUNT_MAP,
        'Expense_Type': EXPENSE_TYPE_MAP,
        'Payment_Status': PAYMENT_MAP, 'Source_System': SOURCE_MAP}
for col, mapping in MAPS.items():
    mapped = df[col].map(mapping)
    bad = df.loc[df[col].notna() & mapped.isna(), col].unique()
    print(col, '- values not mapped:', list(bad))
    df[col] = mapped.fillna(df[col])


Department - values not mapped: []
Region - values not mapped: []
Category - values not mapped: []
Account_Head - values not mapped: []
Expense_Type - values not mapped: []
Payment_Status - values not mapped: []
Source_System - values not mapped: []


In [9]:
# STEP 7: PARSE MIXED DATE FORMATS

DATE_FORMATS = ['%Y-%m-%d', '%d/%m/%Y',
                '%d-%b-%Y', '%B %d, %Y']

def parse_date(text):
    if pd.isna(text):
        return pd.NaT
    for fmt in DATE_FORMATS:
        try:
            return datetime.strptime(text, fmt)
        except ValueError:
            continue
    return pd.NaT

df['Report_Date'] = pd.to_datetime(
    df['Report_Date'].map(parse_date))
print('Invalid / missing dates:',
      df['Report_Date'].isna().sum())

Invalid / missing dates: 34


In [10]:
# STEP 8: VALIDATE THE REPORTING PERIOD

START = pd.Timestamp('2024-01-01')
END = pd.Timestamp('2025-12-31')
outside = (df['Report_Date'].notna()
           & ~df['Report_Date'].between(START, END))
df = df[~outside].copy()
print('After date validation:', df.shape[0])


After date validation: 3950


In [11]:
# STEP 9: CONVERT CURRENCY STRINGS TO NUMBERS

def money_to_float(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    negative = (text.startswith('(')
                or text.startswith('-'))
    text = re.sub(r'(?i)rs\.?|inr|₹', '', text)
    text = re.sub(r'[^0-9.]', '', text)
    if text in ('', '.'):
        return np.nan
    return -float(text) if negative else float(text)

df['Actual_Amount'] = df['Actual_Amount'].map(money_to_float)
df['Budget_Amount'] = df['Budget_Amount'].map(money_to_float)
df['Customer_Count'] = pd.to_numeric(
    df['Customer_Count'].str.replace(',', '', regex=False),
    errors='coerce')

In [12]:
# STEP 10: FIX CATEGORY USING ACCOUNT_HEAD

REVENUE = ['Product Sales', 'Subscription Revenue',
           'Service Revenue', 'Licensing Income']
COGS = ['Raw Materials', 'Direct Labour',
        'Freight & Logistics']
OPERATING = ['Salaries', 'Rent', 'Marketing Spend',
             'Software Licences', 'Travel', 'Utilities',
             'Maintenance', 'Training', 'Recruitment',
             'R&D Expenses', 'Legal & Compliance',
             'Support Tools']
ACCOUNT_INFO = {h: ('Revenue', 'Not Applicable')
                for h in REVENUE}
ACCOUNT_INFO.update({h: ('Expense', 'COGS')
                      for h in COGS})
ACCOUNT_INFO.update({h: ('Expense', 'Operating')
                      for h in OPERATING})

true_cat = df['Account_Head'].map(
    lambda h: ACCOUNT_INFO.get(h, (np.nan, np.nan))[0])
fixes = (true_cat.notna()
         & (df['Category'] != true_cat)).sum()
print('Category values corrected / filled:', fixes)
df['Category'] = true_cat.fillna(df['Category'])



Category values corrected / filled: 207


In [13]:
# STEP 11: STANDARDIZE THE SIGN CONVENTION

is_exp = df['Category'] == 'Expense'
for col in ['Actual_Amount', 'Budget_Amount']:
    df.loc[is_exp, col] = df.loc[is_exp, col].abs()


In [14]:
# STEP 12: REPLACE IMPOSSIBLE VALUES
# Budget_Amount & Actual_Amount will be cleaned at Steps 19 and 16

df.loc[df['Customer_Count'] < 0, 'Customer_Count'] = np.nan
df.loc[df['Budget_Amount'] <= 0, 'Budget_Amount'] = np.nan
df.loc[df['Actual_Amount'] <= 0, 'Actual_Amount'] = np.nan


In [15]:
# STEP 13: CLEAN APPROVED_BY NAMES

df['Approved_By'] = (df['Approved_By']
    .str.replace(r'(?i)^(mr|mrs|ms|miss|dr)\.?\s+', '', regex=True)
    .str.title())


In [16]:
# STEP 14: CLEAN REMARKS TEXT

remarks = (df['Remarks']
           .str.replace(r'<[^>]+>', ' ', regex=True)
           .str.replace(r'#+', ' ', regex=True)
           .str.replace(r'^>+', '', regex=True)
           .str.replace(r'\s+', ' ', regex=True)
           .str.strip())
remarks = remarks.where(remarks != '', np.nan)
df['Remarks'] = (remarks.str[:1].str.upper()
                    + remarks.str[1:])


In [17]:
# STEP 15: REMOVE EXACT DUPLICATE ROWS

print('Exact duplicate rows found:',
      df.duplicated().sum())
df = df.drop_duplicates()
print('After removing duplicates:', df.shape[0])



Exact duplicate rows found: 120
After removing duplicates: 3830


In [18]:
# STEP 16: DROP ROWS MISSING CRITICAL FIELDS

df = df.dropna(subset=['Record_ID', 'Report_Date',
                          'Actual_Amount'])
print('After dropping critical blanks:', df.shape[0])


After dropping critical blanks: 3718


In [19]:
# STEP 17: RESOLVE DUPLICATE RECORD_IDS

df['_order'] = np.arange(len(df))
df['_filled'] = df.notna().sum(axis=1)
df = (df.sort_values(['Record_ID', '_filled', '_order'])
        .drop_duplicates(subset='Record_ID', keep='last')
        .drop(columns=['_order', '_filled']))
print('After resolving duplicate IDs:', df.shape[0])


After resolving duplicate IDs: 3674


In [20]:
# STEP 18: FILL STRUCTURAL AND CATEGORICAL BLANKS

df['Expense_Type'] = df['Expense_Type'].fillna(
    df['Account_Head'].map(
        lambda h: ACCOUNT_INFO.get(h, (np.nan, np.nan))[1]))
df.loc[df['Category'] == 'Expense', 'Customer_Count'] = 0
FILL_UNKNOWN = ['Region', 'Payment_Status',
                'Source_System', 'Approved_By']
for col in FILL_UNKNOWN:
    df[col] = df[col].fillna('Unknown')
df['Remarks'] = df['Remarks'].fillna('No remarks')


In [21]:
# STEP 19: IMPUTE MISSING NUMERIC VALUES

known_region = df['Region'] != 'Unknown'
by_head_region = (
    df[known_region]
    .groupby(['Account_Head', 'Region'])['Budget_Amount']
    .median())
by_head = df.groupby('Account_Head')['Budget_Amount'].median()
missing = df['Budget_Amount'].isna()
keys = list(zip(df.loc[missing, 'Account_Head'],
                df.loc[missing, 'Region']))
df.loc[missing, 'Budget_Amount'] = [
    by_head_region.get(k, by_head[k[0]]) for k in keys]
print('Budget values imputed:', missing.sum())

is_rev = df['Category'] == 'Revenue'
cust_median = (df[is_rev].groupby('Account_Head')['Customer_Count']
               .median())
missing_c = is_rev & df['Customer_Count'].isna()
df.loc[missing_c, 'Customer_Count'] = (
    df.loc[missing_c, 'Account_Head'].map(cust_median).round())
print('Customer counts imputed:', missing_c.sum())


Budget values imputed: 142
Customer counts imputed: 48


In [22]:
# STEP 20: FLAG AND CAP OUTLIERS

log_amt = np.log(df['Actual_Amount'])
group = df['Account_Head']
q1 = log_amt.groupby(group).transform(
    lambda s: s.quantile(0.25))
q3 = log_amt.groupby(group).transform(
    lambda s: s.quantile(0.75))
iqr = q3 - q1
lower = np.exp(q1 - 3 * iqr)
upper = np.exp(q3 + 3 * iqr)
df['Outlier_Flag'] = ((df['Actual_Amount'] < lower)
                      | (df['Actual_Amount'] > upper))
df['Actual_Amount'] = df['Actual_Amount'].clip(
    lower=lower, upper=upper)
print('Outliers flagged and capped:',
      df['Outlier_Flag'].sum())


Outliers flagged and capped: 30


In [23]:
# STEP 21: STANDARDIZE COST_CENTER

DEPT_CC = {'Sales': 'CC-101', 'Marketing': 'CC-102',
           'HR': 'CC-103', 'Operations': 'CC-104',
           'IT': 'CC-105', 'Finance': 'CC-106',
           'Customer Support': 'CC-107', 'R&D': 'CC-108'}
digits = df['Cost_Center'].str.extract(r'(\d{3})')[0]
cc_found = 'CC-' + digits
cc_correct = df['Department'].map(DEPT_CC)
print('Cost centers missing or wrong:',
      (cc_found != cc_correct).sum())
df['Cost_Center'] = cc_correct.fillna(cc_found)


Cost centers missing or wrong: 195


In [24]:
# STEP 22: REBUILD FISCAL_QUARTER

q_raw = pd.to_numeric(
    df['Fiscal_Quarter'].str.extract(r'(?i)^(?:quarter\s*|q)?(\d)')[0])
q_true = df['Report_Date'].dt.quarter
print('Quarters missing or wrong:',
      (q_raw != q_true).sum())
df['Fiscal_Quarter'] = ('Q' + q_true.astype(str) + '-'
                        + df['Report_Date'].dt.year.astype(str))

Quarters missing or wrong: 214


In [25]:
# STEP 23: ENFORCE FINAL DATA TYPES

df['Customer_Count'] = (df['Customer_Count']
                         .round().astype('Int64'))
df['Actual_Amount'] = df['Actual_Amount'].round(2)
df['Budget_Amount'] = df['Budget_Amount'].round(2)
for col in CAT_COLS + ['Fiscal_Quarter', 'Cost_Center']:
    df[col] = df[col].astype('category')
print(df.dtypes)

Record_ID                    str
Report_Date       datetime64[us]
Fiscal_Quarter          category
Department              category
Cost_Center             category
Region                  category
Category                category
Account_Head            category
Expense_Type            category
Actual_Amount            float64
Budget_Amount            float64
Customer_Count             Int64
Payment_Status          category
Source_System           category
Approved_By                  str
Remarks                      str
Outlier_Flag                bool
dtype: object


In [26]:
# STEP 24: SORT, REORDER AND EXPORT

FINAL_COLS = ['Record_ID', 'Report_Date', 'Fiscal_Quarter',
              'Department', 'Cost_Center', 'Region', 'Category',
              'Account_Head', 'Expense_Type', 'Actual_Amount',
              'Budget_Amount', 'Customer_Count', 'Payment_Status',
              'Source_System', 'Approved_By', 'Remarks',
              'Outlier_Flag']
df = (df[FINAL_COLS]
      .sort_values(['Report_Date', 'Department', 'Record_ID'])
      .reset_index(drop=True))

with pd.ExcelWriter(CLEAN_FILE, engine='openpyxl',
                    datetime_format='yyyy-mm-dd') as writer:
    df.to_excel(writer, index=False,
                sheet_name='Cleaned_Data')

assert df['Record_ID'].is_unique
assert df[['Record_ID', 'Report_Date', 'Actual_Amount',
            'Budget_Amount']].notna().all().all()
assert (df['Actual_Amount'] > 0).all()
assert (df['Budget_Amount'] > 0).all()
print('Cleaned dataset saved:', CLEAN_FILE, df.shape)


Cleaned dataset saved: Cleaned_Dataset_Output.xlsx (3674, 17)


In [29]:
# STEP 25: CHECK YOUR RESULT AGAINST THE REFERENCE CLEANED DATASET

reference = pd.read_excel(REFERENCE_FILE, sheet_name=0)
print('Reference shape:', reference.shape)
print('Our result shape:', df.shape)

mine = df.sort_values('Record_ID').reset_index(drop=True)
ref = reference.sort_values('Record_ID').reset_index(drop=True)
try:
    pd.testing.assert_frame_equal(
        mine, ref, check_dtype=False, check_categorical=False)
    print('SUCCESS: our cleaned data matches the reference file.')
except AssertionError as error:
    print('Difference found:', error)
df.info()

Reference shape: (3674, 17)
Our result shape: (3674, 17)
SUCCESS: our cleaned data matches the reference file.
<class 'pandas.DataFrame'>
RangeIndex: 3674 entries, 0 to 3673
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Record_ID       3674 non-null   str           
 1   Report_Date     3674 non-null   datetime64[us]
 2   Fiscal_Quarter  3674 non-null   str           
 3   Department      3674 non-null   str           
 4   Cost_Center     3674 non-null   str           
 5   Region          3674 non-null   str           
 6   Category        3674 non-null   str           
 7   Account_Head    3674 non-null   str           
 8   Expense_Type    3674 non-null   str           
 9   Actual_Amount   3674 non-null   float64       
 10  Budget_Amount   3674 non-null   int64         
 11  Customer_Count  3674 non-null   int64         
 12  Payment_Status  3674 non-null   str           
 13  Source_S